In [ ]:
from datetime import datetime
import pandas as pd
from matplotlib import pyplot as plt
import pycountry
from os import listdir as ls
import numpy as np
import arviz as az
from IPython.display import display, Markdown, Latex
import pycountry_convert as pc
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import OUTPUTS_PATH, DATA_PATH, MOB_LOCATION_SOURCE_MAP, MOB_LOCATION_NAME_MAP, MOBILITY_SMOOTH_PERIOD, G_MOB_LOCATION_CMAP, CONT_CMAP
from emu_renewal.plotting import compare_proc_mob, get_standard_subplot, compare_proc_weighted_gmob
from emu_renewal.utils import get_countries_by_continent, get_countries_with_mob_source, get_job_commits_df, split_list_into_segments
from emu_renewal.selection import get_mob_avail_countries
from emu_renewal.inputs import get_google_mobility, get_world_shp

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "48930936"
all_countries = ls(job_path)
all_countries = [iso3 for iso3 in ls(job_path) if (job_path / iso3 / "oxcgrt").exists()]  # Some countries run for mobility baseline but no CGRT data

countries_by_cont = get_countries_by_continent(all_countries)
notes = {
    "g_mob": "Mobility values presented as Google data divided by 100, plus one.",
    "fb_visited_mob": "Mobility values presented as one plus Facebook data.",
    "fb_singletile_mob": "Mobility values presented as one minus Facebook data.",
}

In [ ]:
from emu_renewal.inputs import get_oxcgrt, get_smoothed_trunc_g_mob, get_google_mobility, get_weight_posts

In [ ]:
cont_countries = countries_by_cont["AF"]
country = cont_countries[1]
c_path = job_path / country

In [ ]:
get_weight_posts(c_path, "oxcgrt")

In [ ]:
fig, axes = get_standard_subplot(len(cont_countries), 4)
flat_axes = axes.ravel()
for c, iso3 in enumerate(cont_countries):
    ax = flat_axes[c]
    ax.set_title(pycountry.countries.lookup(iso3).name)
    proc_samples = pd.read_hdf(job_path / iso3 / "no_mob/spaghetti.h5")["process"]
    centiles = proc_samples.quantile([0.025, 0.5, 0.975], axis=1).T
    ax.plot(centiles.index, centiles[0.5], label="process", color="navy", linewidth=2.0)

    idata = az.from_netcdf(job_path / iso3 / "oxcgrt/idata_filtered.nc")
    params = idata.posterior["mob_weights"].to_dataframe().unstack(level=-1)

fig.tight_layout()

In [ ]:
# display(Markdown("# Individual mobility metric comparisons (Google and Facebook)\n\n"))
# for mob_type in MOB_LOCATION_SOURCE_MAP:
#     mob_source = MOB_LOCATION_SOURCE_MAP[mob_type]
#     mob_name = MOB_LOCATION_NAME_MAP[mob_type]
#     display(Latex(r"\newpage"))
#     section_title = f"## {mob_name} mobility metric comparison\n\n"
#     display(Markdown(section_title))
#     display(Markdown(notes[mob_source]))
#     mob_avail_countries = get_countries_with_mob_source(job_path, mob_source)
#     for cont in CONT_CMAP:
#         cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
#         display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
#         for countries in split_list_into_segments(cont_countries, 9):
#             display(compare_proc_mob(job_path, countries, 4, mob_type))

display(Markdown("# Composite Google mobility comparison\n\n"))
for cont in CONT_CMAP:
    display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
    n_cols = 3 if cont in ["NA", "SA"] else 4
    mob_avail_countries = get_countries_with_mob_source(job_path, "g_mob")
    cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
    display(compare_proc_weighted_gmob(job_path, cont_countries, 200, n_cols))

{{< pagebreak >}}

# Commits used for analyses
For reproducibility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_job_commits_df(job_path, all_countries).to_markdown())